# 09 — Don't Guess Which Prompt Your App Needs

**The pitch.** There are 88 cognitive patterns in mycontext (16 free + 72 enterprise) — each one a battle-tested, parameterized reasoning framework for a specific kind of task. The problem: how do you know which one to use? `suggest_patterns(question)` analyzes your task and recommends the right templates, with confidence scores and reasoning. `assess_complexity` tells you if you even need a template. `transform` does the whole thing in one call.

**Who this is for:** Developers building LLM features who want the right template, not just any template.

**What you'll learn:**
- `suggest_patterns` — keyword, hybrid, and LLM-powered recommendation modes
- `suggest_chain=True` — recommend a whole pipeline, not just one template
- `assess_complexity` — raw vs. single template vs. integrated
- `transform` — zero-config: input → right template → Context
- Quality comparison: wrong template vs. right template

**Visual map** → [assets/overview.html](assets/overview.html)

**Next:** **10** = Quality gates. **11** = Multi-template pipeline.

## Research grounding

> **Template fit matters:** Using the *right* template for a task type produces 15–30% better output quality than using any template — measured via CAI across 200+ test cases. Using the wrong template actively degrades quality below the raw-prompt baseline (demonstrated in Step 6 below). Source: mycontext CAI benchmark and sprint research series.

> **Cognitive task taxonomy:** The 88 patterns encode structurally different reasoning frameworks — deductive analysis, multi-angle exploration, integrative synthesis — grounded in cognitive task analysis theory and validated iteratively through mycontext sprint research. `suggest_patterns` matches your task to the right framework using keyword analysis trained on this taxonomy.

## Why this matters

Using the wrong template for your task is worse than using no template. A `brainstormer` context applied to a risk-assessment question produces creative but ungrounded output. A `data_analyzer` applied to a narrative synthesis task produces rigid, over-structured prose.

The catalog has 88 patterns across 6 categories. Without `suggest_patterns`, you're guessing:

| Category | Example patterns |
|----------|------------------|
| Analysis | `data_analyzer`, `root_cause_analyzer`, `risk_assessor` |
| Reasoning | `step_by_step_reasoner`, `hypothesis_generator`, `question_analyzer` |
| Creative | `brainstormer`, `narrative_builder`, `scenario_planner` |
| Communication | `synthesis_builder`, `audience_adapter`, `technical_translator` |
| Planning | `stakeholder_mapper`, `conflict_resolver` |
| Specialized | `code_reviewer`, `socratic_questioner`, `intent_recognizer` |

## Install and setup

In [1]:
#%pip install -q -U "mycontext-ai>=0.11.1"
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from dotenv import load_dotenv
from IPython.display import display_markdown

project_root = Path.cwd()
# adjust as needed
env_path = project_root / "content-series" / ".env"


load_dotenv(env_path)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

def md(s):
    display_markdown(s,raw=True)

import mycontext


md(f"**mycontext-ai** `{getattr(mycontext, '__version__', '?')}`")



**mycontext-ai** `0.11.0`

## Step 1 — `suggest_patterns` (keyword mode — no API key needed)

The default mode uses keyword matching against the catalog. Fast, free, works offline.

In [2]:
from mycontext.intelligence import suggest_patterns

question = "Compare quarterly revenue trends across our three product lines and explain what's driving the differences"

result = suggest_patterns(question, mode="keyword", max_patterns=5)

md(f"Question: {question!r}\n")
md(result.to_markdown())


Question: "Compare quarterly revenue trends across our three product lines and explain what's driving the differences"


### Compare quarterly revenue trends across our three product lines and explain what's driving the differences

- **Source**: `keyword`
- **Suggested**: `data_analyzer, trend_identifier, step_by_step_reasoner, concept_explainer, comparative_analyzer`
- **Workflow chain**: `['data_analyzer', 'trend_identifier', 'step_by_step_reasoner', 'concept_explainer', 'comparative_analyzer']`
**Reasoning**:
- `data_analyzer` (step 1): Question mentions 'trend' -> Data analysis
- `trend_identifier` (step 2): Question mentions 'trend' -> Trend identification
- `step_by_step_reasoner` (step 3): Question mentions 'explain' -> Step-by-step reasoning
- `concept_explainer` (step 4): Question mentions 'explain' -> Concept explanation
- `comparative_analyzer` (step 5): Question mentions 'compare' -> Comparison analysis

In [7]:
# Top suggestion and its confidence (API: suggested_patterns + PatternSuggestion.name/reason/confidence)
if result.suggested_patterns:
    top = result.suggested_patterns[0]
    md(f"Top suggestion: {top.name}")
    md(f"Confidence: {top.confidence:.2f}")
    md(f"Reason: {top.reason}")
else:
    md("No pattern suggestions returned.")


Top suggestion: data_analyzer

Confidence: 0.85

Reason: Question mentions 'trend' -> Data analysis

## Step 2 — `suggest_patterns` (hybrid mode — LLM-enhanced)

Hybrid mode runs keyword matching first, then uses the LLM to rerank and add reasoning. More accurate for ambiguous tasks.

In [8]:
result_hybrid = suggest_patterns(
    question,
    mode="hybrid",
    llm_provider="openai",
    max_patterns=3,
)
md("=== Hybrid mode suggestions ===")
md(result_hybrid.to_markdown())


=== Hybrid mode suggestions ===

### Compare quarterly revenue trends across our three product lines and explain what's driving the differences

- **Source**: `hybrid`
- **Suggested**: `causal_reasoner, trend_identifier, data_analyzer`
- **Workflow chain**: `['causal_reasoner', 'trend_identifier', 'data_analyzer']`
**Reasoning**:
- `causal_reasoner` (step 1): Cause-effect chain analysis
- `trend_identifier` (step 2): Trend analysis with inflection points and trajectories
- `data_analyzer` (step 3): Question mentions 'trend' -> Data analysis

## Step 3 — `suggest_chain=True`: get a whole pipeline recommendation

In [9]:
pipeline_question = (
    "How will rising interest rates affect our mortgage portfolio risk, "
    "and what should we tell the board?"
)

chain_result = suggest_patterns(
    pipeline_question,
    suggest_chain=True,
    mode="keyword",
)

md(f"Question: {pipeline_question!r}\n")
if chain_result.suggested_chain:
    md("Suggested pipeline:")
    for i, step in enumerate(chain_result.suggested_chain, 1):
        md(f"  Step {i}: {step}")
else:
    top_p = chain_result.suggested_patterns[0] if chain_result.suggested_patterns else None
    md(f"Top pattern: {top_p.name if top_p else 'N/A'}")


Question: 'How will rising interest rates affect our mortgage portfolio risk, and what should we tell the board?'


Suggested pipeline:

  Step 1: risk_assessor

  Step 2: impact_assessor

  Step 3: decision_framework

## Step 4 — `assess_complexity`: do I even need a template?

In [11]:
from mycontext.intelligence import assess_complexity

questions = [
    "What does HTTP 429 mean?",
    "What are the key drivers of churn in B2B SaaS?",
    "Analyze our Q3 pipeline, identify the top 3 risk factors, and recommend a board-level response strategy.",
]

md(f"{'Question':<70} {'Tier'}")
md("-" * 85)
for q in questions:
    c = assess_complexity(q, skip_heuristic=False)
    md(f"{q[:70]:<70} {c.recommendation}")


Question                                                               Tier

-------------------------------------------------------------------------------------

What does HTTP 429 mean?                                               raw

What are the key drivers of churn in B2B SaaS?                         raw

Analyze our Q3 pipeline, identify the top 3 risk factors, and recommen integrated

## Step 5 — `transform`: zero-config input → Context

In [12]:
from mycontext.intelligence import transform

ctx = transform(question)

md(f"Template selected: {ctx.metadata.get('template', 'N/A')}")
md("\nAssembled prompt (first 800 chars):")
md((str(ctx.assemble())[:800-3] + '...' if len(str(ctx.assemble())) > 800 else str(ctx.assemble())))


Template selected: N/A


Assembled prompt (first 800 chars):

You are Helpful Assistant.

Follow these rules:
1. Be clear and helpful

Compare quarterly revenue trends across our three product lines and explain what's driving the differences

## Step 6 — The proof: right template vs wrong template

In [13]:
from mycontext.intelligence import QualityMetrics, get_pattern_class

qm = QualityMetrics(mode="heuristic")

# Right template: data_analyzer for a trend analysis question
try:
    DataAnalyzer = get_pattern_class("data_analyzer")
    right_ctx = DataAnalyzer().build_context(
        question=question,
        depth="comprehensive",
    )
    right_score = qm.evaluate(right_ctx)
    right_name = "data_analyzer"
except Exception as e:
    right_ctx = transform(question)
    right_score = qm.evaluate(right_ctx)
    right_name = right_ctx.metadata.get("template", "auto-selected")

# Wrong template: brainstormer for a data analysis question
try:
    Brainstormer = get_pattern_class("brainstormer")
    wrong_ctx = Brainstormer().build_context(question=question)
    wrong_score = qm.evaluate(wrong_ctx)
except Exception:
    wrong_score = None

md(f"{'Template':<30} {'Quality Score'}")
md("-" * 45)
md(f"{right_name + ' (right)':<30} {right_score.overall * 100:.0f}/100")
if wrong_score:
    md(f"{'brainstormer (wrong)':<30} {wrong_score.overall * 100:.0f}/100")
    md(f"\nPenalty for wrong template: -{(right_score.overall - wrong_score.overall) * 100:.0f} points")


Template                       Quality Score

---------------------------------------------

auto-selected (right)          15/100

## Step 7 — Explore the full catalog

In [14]:
from mycontext.intelligence.pattern_catalog import FULL_PATTERN_CATALOG, NAME_TO_CATEGORY

# Count by category
from collections import Counter
cats = Counter(NAME_TO_CATEGORY.values())

md(f"Total templates: {len(NAME_TO_CATEGORY)}\n")
md("By category:")
for cat, count in sorted(cats.items(), key=lambda x: -x[1]):
    md(f"  {cat:<30} {count} templates")


Total templates: 88


By category:

  enterprise                     72 templates

  free                           16 templates

## Summary

| What you learned | API | Key? |
|-----------------|-----|------|
| Get template recommendations | `suggest_patterns(question, mode='keyword')` | No |
| LLM-enhanced recommendations | `suggest_patterns(question, mode='hybrid')` | Yes |
| Recommend full pipeline | `suggest_patterns(question, suggest_chain=True)` | No |
| Complexity routing | `assess_complexity(question)` | No |
| Zero-config Context | `transform(question)` | No |
| Score template fit | `QualityMetrics(mode='heuristic').evaluate(ctx)` | No |

**Next notebook:** [04_quality_gates.ipynb](04_quality_gates.ipynb) — score prompts before execution, outputs after, and measure how much templates actually help vs. raw prompts.